# Individual Neurons Analysis *✧･ﾟ: *✧･ﾟ:*✧･ﾟ: *✧･ﾟ:

This notebook process the OSI and DSI analysis for the Calcium Imaging data for the project:
"Visual Cortex sensory coding on a BTBR model of autism".

---
### Expected File Organization
```
Calcio/
├── BTBR_V1/                  ← File name defines the genotype name
│   ├── animal_fecha/
│   │   ├── *reg*.txt         ← Scope data
│   │   ├── *pas*.txt         ← Stimuli data (psychopy)
│   │   └── *traces*.csv      ← Calcium traces
│   └── ...
└── C57_V1/
    └── ...
```

<div style="background-color:##99f720; border-left: 6px solid #99f720; padding: 10px; border-radius: 4px;">
    <h4 style="margin-top: 0;">What happens if a folder name isn't recognized?</h4>
    If a group folder is named something like BTBR_V1_copy it won't either match "BTBR" or "C57". When that happens `detect_genotype` raises an error instead of skipping the folder silently. Not doing it wouldn't break anything immediately, but that animal would silently drop out of the analysis (you might report N=12 when only 11 were actually processed, and nobody would notice until the sample sizes stop adding up.). The tradeoff is that if the pipeline stops, someone has to check it by hand. 
</div>

- Each line in the scope log contains two different clocks: `12:17:32.052703` refers to the moment the computer received the serial data and can have a jitter. and `16692` that is the internal cronometer from the scope hardware.
- The PC timestamp (por qué tiene jitter), while the hardware counter
- Looking at consecutive frame events in real data.

12:17:32   .   052703        .   16692            .  23    .  1
PC clock       PC micros         HW counter (ms)     flag    channel

## 1. Import dependencies and setup
Standard scientific Python stack, plus re for validating raw log line formats before parsing.

In [ ]:
import re 
import numpy as np
from pathlib import Path

In [ ]:
# ₊˚ʚ ROUTES ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂
DATA_ROOT = r'D:\AL-205\Nicole\Calcio\Nueva carpeta'  # <── change this

# Keywords for detecting files (case-insensitive)
SCOPE_KEYWORD  = 'reg'     # TXT scope data
STIM_KEYWORD   = 'pas'     # TXT stimuli
TRACES_KEYWORD = 'traces'  # CSV calcium traces

# group folders → genotype
GENOTYPE_FOLDERS = {
    'BTBR': 'BTBR',
    'C57':  'C57',
}

# ₊˚ʚ  AQUISICION ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁
SAMPLING_RATE = 5           # Hz
MS_PER_FRAME  = 1000 / SAMPLING_RATE  # 200 ms nominal — measured ~190ms in raw hardware counter, see Section 0. Pending resolution

# ₊˚ʚ  STIM WINDOWS ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁
SEC_BEFORE    = 2   # baseline window, matches stimulus protocol (see Methods)
SEC_AFTER     = 4   # stimulus window, matches trial duration
FRAMES_BEFORE = int(SEC_BEFORE * SAMPLING_RATE)   # 10
FRAMES_AFTER  = int(SEC_AFTER  * SAMPLING_RATE)   # 20
TOTAL_FRAMES  = FRAMES_BEFORE + FRAMES_AFTER       # 30

# Response window for used to compute per-angle mean response (tuning curve ➝ OSI/DSI)
RESP_START = FRAMES_BEFORE      # frame 10
RESP_END   = TOTAL_FRAMES       # frame 30

# ₊˚ʚ  THRESHOLD ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁
RESPONSIVE_STD   = 3    # STDs from baseline for a responsive neuron TODO: cite source
OSI_THRESH       = 0.7  # threshold for orientation-selective neuron, per TODO: cite source

# ₊˚ʚ  OUTPUT ⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂⠂⠁⠈⠂⠄⠄⠂⠁⠁⠂⠄⠄⠂⠁⠁⠂
OUTPUT_DIR = Path(DATA_ROOT) / 'pipeline_output_real'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup ready')
print(f'Output → {OUTPUT_DIR}')